# 탐색적 데이터 분석

pandas, matplotlib, seaborn, numpy를 이용해서 train 데이터셋의 컬럼들을 탐색하고 분석함

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../../data/iowa-housing/train.csv")
DATA_PATH.exists()

In [ ]:
df = pd.read_csv(DATA_PATH)
df.shape

In [ ]:
df.describe()

In [ ]:
df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

sns.set_theme(font="Malgun Gothic")

In [ ]:
# SalePrice 히스토그램
plt.figure(figsize=(8,5))

sns.histplot(df["SalePrice"], bins=50, kde=True)

plt.title("SalePrice 분포")
plt.xlabel("판매가 ($)")
plt.ylabel("집 수")
plt.show()

In [ ]:
sale_price_col = df.SalePrice
sale_price_col.describe()

In [ ]:
plt.figure(figsize=(8, 2))

sns.boxplot(x=df["SalePrice"])

plt.title("SalePrice 박스 플롯 (이상치 확인)")
plt.show()

In [ ]:
import numpy as np 

print("원본 skew  :", df["SalePrice"].skew().round(3))
print("log변환 skew :", np.log1p(df["SalePrice"]).skew().round(3))

plt.figure(figsize=(8,5))
sns.histplot(np.log1p(df["SalePrice"]), bins=50, kde=True)
plt.title("log(SalePrice) 분포")
plt.xlabel("log(판매가)")
plt.ylabel("집 수")
plt.show()

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
print(len(num_cols), "개")
num_cols

In [ ]:
n = len(num_cols)
ncols = 4
nrows = -(-n // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 3))

for col, ax in zip(num_cols, axes.flatten()):
    sns.histplot(df[col], bins=30, kde=False, ax=ax)
    ax.set_title(col, fontsize=10)

for ax in axes.flatten()[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
check_cols = ["LotFrontage", "YearBuilt", "YearRemodAdd", "TotalBsmtSF", 
              "1stFlrSF", "GrLivArea", "GarageArea", "SalePrice"]
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for col, ax in zip(check_cols, axes.flatten()):
    sns.histplot(df[col], bins=40, kde=True, ax=ax)
    skew = df[col].skew()
    ax.set_title(f"{col}\nskew={skew:.2f}")

plt.tight_layout()
plt.show()

In [ ]:
# 범주형 컬럼 전부 뽑기
cat_cols = df.select_dtypes(include="str").columns.tolist()
print(len(cat_cols), "개")
cat_cols

In [ ]:
df[cat_cols].nunique().sort_values(ascending=False)

In [ ]:
# 고유값 개수로 범주형을 두 그룹으로 나누기
nunq = df[cat_cols].nunique()

small_cat = nunq[nunq <= 8].index.tolist()
large_cat = nunq[nunq > 8].index.tolist()

print(f"적음({len(small_cat)}개):", small_cat)
print(f"많음({len(large_cat)}개):", large_cat)

In [ ]:
n = len(small_cat)
ncols = 4
nrows = -(-n // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 3.5))

for col, ax in zip(small_cat, axes.flatten()):
    sns.countplot(data=df, x=col, ax=ax)
    ax.set_title(col, fontsize=10)
    ax.tick_params(axis="x", rotation=45)

for ax in axes.flatten()[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# 가장 흔한 값이 전체의 몇 %인가 (쏠림 정도)
for col in cat_cols:
    top_ratio = df[col].value_counts(normalize=True).iloc[0]
    print(f"{col:15s} {top_ratio:.0%} (최빈값: {df[col].mode()[0]})")

In [ ]:
# 비율 + 최빈값을 한 표로
skew_cat = pd.DataFrame({
    "top_ratio": df[cat_cols].apply(lambda s: s.value_counts(normalize=True).iloc[0]),
    "top_value": df[cat_cols].apply(lambda s: s.value_counts().index[0]),
}).sort_values("top_ratio", ascending=True)

skew_cat

In [ ]:
# 컬럼별 결측 비율 (결측 있는 것만, 많은 순)
missing = df.isnull().mean()
missing = missing[missing > 0].sort_values(ascending=False)
missing

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=missing.values, y=missing.index)
plt.title("컬럼별 결측 비율")
plt.xlabel("결측 비율")
plt.show()

In [ ]:
# 결측 있는 컬럼만 추출 → 범주형/수치형 분리
miss_cols = df.columns[df.isnull().any()]
miss_cat = df[miss_cols].select_dtypes("str").columns
miss_num = df[miss_cols].select_dtypes("number").columns

# 결측 있는 범주형 분포
n = len(miss_cat)
fig, axes = plt.subplots(-(-n // 3), 3, figsize=(16, -(-n // 3) * 3))

for col, ax in zip(miss_cat, axes.flatten()):
    df[col].value_counts(dropna=False).plot(kind="bar", ax=ax) 
    ax.set_title(f"{col} (결측 {df[col].isnull().mean():.0%})")

for ax in axes.flatten()[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# 결측 있는 수치형 분포
n = len(miss_num)
fig, axes = plt.subplots(-(-n // 3), 3, figsize=(16, -(-n // 3) * 3))

for col, ax in zip(miss_num, axes.flatten()):
    sns.histplot(df[col], bins=30, ax=ax)
    ax.set_title(f"{col} (결측 {df[col].isnull().mean():.0%})")

for ax in axes.flatten()[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
df[miss_num].isnull().sum()

In [ ]:
for prefix in ["Garage", "Bsmt", "Pool", "MasVnr", "Fireplace"]:
    cols = [c for c in df.columns if prefix in c]
    print(f"{prefix}: {cols}")

for prefix in ["Garage", "Bsmt", "Pool", "MasVnr", "Fireplace"]:
    cols = [c for c in df.columns if prefix in c]
    print(f"=== {prefix} ===")
    print(df[cols].isnull().sum())
    print()

In [ ]:
# MasVnr 확인
print("=== MasVnrType이 NaN인 집의 MasVnrArea ===")
print(df.loc[df["MasVnrType"].isnull(), "MasVnrArea"].value_counts(dropna=False).head())

print("Type만 NaN:", (df["MasVnrType"].isnull() & df["MasVnrArea"].notnull()).sum())
print("Area만 NaN:", (df["MasVnrType"].notnull() & df["MasVnrArea"].isnull()).sum())
print("둘 다 NaN:", (df["MasVnrType"].isnull() & df["MasVnrArea"].isnull()).sum())

In [ ]:
print(df["GarageYrBlt"].dtypes)
print(df["GarageArea"].dtypes)
print(df["MasVnrArea"].dtypes)
print(df["MasVnrType"].dtypes)

# GarageYrBlt와 GarageArea의 결측치 확인
print(df.loc[df["GarageYrBlt"].isnull(), "GarageArea"].value_counts())

# MasVnrArea와 MasVnrType의 결측치 확인
print(df.loc[df["MasVnrArea"].isnull(), "MasVnrType"].value_counts(dropna=False))

In [ ]:
print("LotFrontage 결측:", df["LotFrontage"].isnull().sum(), "개")
sns.histplot(df["LotFrontage"], bins=40, kde=True)
plt.title("LotFrontage 분포 (값 있는 것만)")
plt.show()

In [ ]:
# 수치형 변수들기리의 상관계수
corr = df.corr(numeric_only=True)

plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("수치형 변수 상관계수 히트맵")
plt.show()

In [ ]:
# SalePrice 상관계수
corr_target = corr["SalePrice"].sort_values(ascending=False)
corr_target

In [ ]:
# 상위 4개 변수와 SalePrice의 관계
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# OverallQual: 등급별 → 박스플롯
sns.boxplot(data=df, x="OverallQual", y="SalePrice", ax=axes[0, 0])
axes[0, 0].set_title("OverallQual vs SalePrice")

# GrLivArea: 연속 → 산점도
sns.scatterplot(data=df, x="GrLivArea", y="SalePrice", ax=axes[0, 1])
axes[0, 1].set_title("GrLivArea vs SalePrice")

# TotalBsmtSF: 연속 → 산점도
sns.scatterplot(data=df, x="TotalBsmtSF", y="SalePrice", ax=axes[1, 0])
axes[1, 0].set_title("TotalBsmtSF vs SalePrice")

# GarageArea: 연속 → 산점도
sns.scatterplot(data=df, x="GarageArea", y="SalePrice", ax=axes[1, 1])
axes[1, 1].set_title("GarageArea vs SalePrice")

plt.tight_layout()
plt.show()